---
title: "Reading the wt CLI from the outside in"
date: "2026-08-22"
categories: ["meta", "dev"]
toc: true
---

`wt` is the command-line interface used to navigate the `watchtower` (⛫) workspace. 
That workspace is also the source for this entire website: its notebooks hold
the notes, articles, and courses, while `wt` provides the commands for reading,
editing, running, and rendering them.

## General workflows

`wt` is easiest to understand as a path from a shell command to a notebook
cell. Most commands follow the same small loop:

```text
parse command -> find a file -> read a notebook -> inspect or change cells
```

The interesting details are in the edges of that loop: how names resolve,
how notebook JSON becomes agent-readable text, how a cell is changed without
discarding its outputs, and how course problems use tags instead of a second
database.

This walkthrough follows three representative journeys:

1. `wt cat`, which reads a notebook for an agent;
2. `wt edit-cell` and `wt run`, which write notebook state back; and
3. `wt problem` and `wt solution`, which treat course notebooks as a small
   structured content system.

Around that center are import, scaffolding, rendering, resume, vault, and ML
plotting helpers. They are part of the same package, but they do not need to
be forced into one architectural story.


## 1. Start at `cli.py`: the command map

The CLI is built with Typer. The top-level app owns the command names, while
`new` and `vault` are mounted sub-applications:

```python
app = typer.Typer(
    name="wt",
    help="Personal notes, articles, courses, and projects system.",
    no_args_is_help=True,
)

new_app = typer.Typer(name="new", help="Scaffold new artifacts.")
app.add_typer(new_app)

vault_app = typer.Typer(name="vault", help="Manage secrets in OS keyring.")
app.add_typer(vault_app)
```

The command functions mostly translate CLI arguments into calls to a module
function. For example, `cat` imports `notebook` inside its body, selects the
effective read limit, calls `cat_notebook`, and prints the result. Imports are
kept close to the command that needs them, so `wt vault ls` does not need to
load the notebook execution stack.

The current command map includes:

- repository navigation: `map`, `ls`, `find`, `count`, and `kernels`;
- notebook operations: `cat`, `output`, `diff`, `run`, `edit-cell`,
  `append-cell`, `insert-cell`, `remove-cell`, `clear-outputs`, and `tag`;
- course operations: `problem`, `solution`, `hint`, `check`,
  `add-exercise`, and `solution-set`;
- creation and integration: `new`, `import`, `render`, `resume`, and `docs`;
- secrets: `vault set`, `get`, `rm`, `ls`, and `export`.

The two console scripts in `pyproject.toml`, `wt` and `watchtower`, both point
at `watchtower.cli:main`. `main()` is also the final error boundary: selected
file and argument errors become a readable message and exit status 1.



## 2. From a name to a notebook

The first real question for a notebook command is not “what cell?” It is
“which file does this name mean?” That logic lives in
`inspect.resolve_ipynb()`.

Its resolution ladder is short:

```python
def resolve_ipynb(name: str) -> Path:
    maybe = Path(name)
    if maybe.exists() and maybe.suffix == ".ipynb":
        return maybe.resolve()

    parts = Path(name).parts
    if parts and parts[0] in {"notes", "articles", "courses"}:
        path = NB_DIR.joinpath(*parts)
        if path.suffix != ".ipynb":
            path = path.with_suffix(".ipynb")
        if path.exists():
            return path.resolve()
    if parts and parts[0] == NB_DIR.name and len(parts) > 1:
        path = Path(*parts)
        if path.suffix != ".ipynb":
            path = path.with_suffix(".ipynb")
        if path.exists():
            return path.resolve()

    for base in CONTENT_DIRS:
        for path in base.rglob(f"{name}.ipynb"):
            if ".ipynb_checkpoints" not in path.parts:
                return path

    raise FileNotFoundError(...)
```

That gives the CLI three useful forms:

```text
nb/notes/005-wt-src-walkthrough.ipynb   explicit path
nb/notes/005-wt-src-walkthrough         tier plus stem
005-wt-src-walkthrough               bare stem
```

`paths.py` supplies the content-directory constants and the absolute
`ROOT_PATH` used for repository-anchored generated artifacts. The content
directories themselves are relative paths, which is why the normal workflow
starts in the repository root and why the tests change into temporary repo
directories.

The rest of `inspect.py` builds on the same filesystem view. `wt map` returns
the repository structure as JSON, `ls` lists notebooks while excluding index
and checkpoint files, and `find` searches notebook sources. `find` uses
`rg` as a quick candidate filter when it is available, then parses matching
notebooks to report the cell index and matching line.


## 3. `notebook.py`: the shared cell layer

The core notebook path can be pictured like this:

```{mermaid}
flowchart LR
    name["user-supplied name"] --> resolve["resolve_ipynb"]
    resolve --> read["nbformat.read"]
    read --> inspect["render cells"]
    read --> mutate["change cells"]
    mutate --> write["nbformat.write"]
    inspect --> stdout["print Markdown-like text"]
```

There is no long-lived `Notebook` object. Each CLI process resolves a path,
reads a `NotebookNode`, performs one operation, and exits. That makes the
state easy to find: it is in the `.ipynb` file.

`cat_notebook()` is the read side. It can render every cell, or select cells
by:

- an index, including Python-style ranges such as `0:3` and negative indices;
- a Jupyter tag; or
- a leading Quarto `#| label:` pragma.

`_render_cell()` adds a small header such as `> cell 4 [code]`, renders code
cells inside a Python fence, and leaves Markdown sources readable as Markdown.
The same renderer is reused by `wt diff`, which is why notebook diffs show
content instead of a wall of JSON.

The write functions are deliberately small: `edit_cell`, `append_cell`,
`insert_cell`, `remove_cell`, `clear_outputs`, and `tag_cell`. They validate
locators, enforce the 20,000-character source limit, mutate the in-memory
notebook, and write it back through `nbformat`. `edit_cell` changes the
selected cell's source, so its existing outputs and metadata remain attached
to that cell. `remove_cell` deletes matching indices in reverse order so the
remaining positions do not shift under the loop.


## 4. Why `cat` is shaped for agent reads

The normal notebook representation is excellent for Jupyter and awkward for
an agent: cell sources, metadata, execution counts, and output payloads are
all nested in JSON. `cat` creates a compact text view instead.

For example, a selected code cell looks conceptually like this:

````text
> cell 4 [code] tags:example

```{python}
print("hello")
```
````

The command has a few features that make long notebooks manageable:

- source reads default to 4,096 characters per cell;
- `--offset` and `--limit` make a long cell readable in successive slices;
- `--context N` includes nearby cells and marks them as context;
- `--with-outputs` appends stored outputs with their own headers; and
- `--decode` is an explicit opt-in for readable course solutions.

Output rendering is intentionally conservative. Text and errors are shown;
PNG, JPEG, and SVG payloads are summarized rather than expanded into base64.
When an agent needs the actual image, `outputs.py` provides a structured
`CellOutput` interface and `wt output` decodes image bytes into `.tmp`.

The same plain representation powers `wt diff`. `diff_notebook()` reads the
base version with `git show`, renders both versions through the notebook
renderer, and decodes solution cells so a change to a hidden solution is
still understandable in the diff.


## 5. The write path and the execution path

An edit is a simple notebook round trip:

```python
path = resolve_ipynb(name)
nb = read_notebook(path)
cell_index = _resolve_unique_cell(nb, ...)
nb["cells"][cell_index]["source"] = source
nbformat.write(nb, path)
```

The important part is that the operation changes a loaded notebook object,
not a hand-edited JSON string. Existing metadata and outputs survive because
they are still present when `nbformat.write()` serializes the object. The
file is still the source of truth, so `wt clear-outputs` and `wt run` visibly
change the stored notebook state.

`wt run` takes the next step by using `nbclient`:

```text
read notebook -> start kernel -> execute code cells -> store outputs -> write notebook
```

A normal run executes all code cells in one kernel. Errors are stored as
inline error outputs and execution continues. `wt run --index N` executes
only that cell in a fresh kernel, which is useful for testing a self-contained
cell and intentionally does not provide state from earlier cells. `wt kernels`
lists the installed kernels that can be passed with `--kernel`.

The repository is configured so Quarto renders stored inline outputs without
re-executing code. That makes `wt run` the explicit refresh operation: a
render can faithfully show outputs that were produced earlier, including
outputs that are now stale.


## 6. Course problems: structure carried by tags

Course problems live inside chapter notebooks. There is no separate problem
database. The cell layout is the data model:

```text
problem cell [problem, 07-3]
        |
        +--> optional starter code cell
        |
        +--> solution cell [solution, 07-3]
```

The shared id is what connects the cells. `problems.resolve_problem()` first
resolves the chapter, then finds the problem cell with the matching id tag,
looks for an immediately following starter code cell, and finds the matching
solution cell. Chapter locators can be numeric (`7.3`, `07-3`, `07 3`) or
name-based (`projection 3`).

Solutions are stored in code cells so they can carry Quarto cell options, but
they are not meant to execute. `obfuscate.wrap()` leaves the hiding options
literal and applies ROT13 to letters plus ROT5 to digits to the solution body.
It also prefixes non-empty encoded lines with `#`, making the source both
hidden from the rendered page and visually distinct in JupyterLab.

The transform is reversible, not cryptographic. The public workflows make
that explicit:

```text
wt add-exercise    plaintext in -> wrapped solution cell
wt solution-set   plaintext in -> replace wrapped solution cell
wt solution       wrapped cell -> decoded solution
wt hint           wrapped cell -> progressive decoded hint
wt check          notebook -> pairing and encoding warnings
```

This is a nice example of using notebook metadata as an application schema:
the content stays portable, while `wt` provides the conventions and checks
around it.


## 7. The supporting modules

The notebook layer is the center of the walkthrough, but the package has
several useful neighbors.

| Area | Modules | Main job |
|---|---|---|
| Create content | `scaffold.py` | Build note, article, course, chapter, section, and project stubs |
| Import content | `convert.py` | Copy external notebooks, preserve outputs, normalize kernelspecs, and register course chapters |
| Render content | `render.py` | Invoke Quarto for PDFs or a blocking site preview |
| Build the resume | `resume.py` | Turn YAML and Jinja templates into the web page, LaTeX source, and PDF |
| Store secrets | `vault.py` | Use the OS keyring with a local index of available key names |
| ML notebook tools | `core/` | Provide plotting primitives and a Python/NumPy/PyTorch seed helper |
| Discover kernels | `kernels.py` | List installed Jupyter kernels for `wt run --kernel` |

`scaffold.py` is where content creation meets the site configuration. It uses
`ruamel.yaml` to register courses and chapters in `_quarto.yml`, preserving
the YAML document while adding sidebar entries. `convert.py` reuses that
registration path when importing a chapter.

The other modules cross external boundaries. Quarto renders, `pdflatex`
builds the resume, `uv` creates projects, and the OS keyring stores secrets.
The Python package supplies the arguments and file conventions; those tools
still do the work.



## 8. Read the tests alongside the source

The tests are the best way to see which details are meant to stay stable.
They use temporary repositories for filesystem operations, so notebook edits
and sidebar changes can be tested without touching the real knowledge base.

The most useful reading pairs are:

| Source | Tests | What the pair teaches |
|---|---|---|
| `notebook.py` | `test_notebook.py` | Cell selectors, slices, writes, output clearing, and readable diffs |
| `inspect.py` | `test_inspect.py` | Repository listing, resolution, and source search |
| `scaffold.py` | `test_scaffold.py` | Notebook stubs and sidebar registration |
| `convert.py` | `test_convert.py` | Import normalization and duplicate-title handling |
| `problems.py` | `test_problems.py` | Problem ids, solution wrapping, hints, and checks |
| `execute.py` | `test_execute.py` | Real-kernel execution and persisted outputs |
| `outputs.py` | `test_outputs.py` | Text/image normalization and image extraction |
| `cli.py` | `test_cli.py` | The command boundary through Typer's test runner |

The suite covers the notebook and course center directly, and the execution
tests launch a real Python kernel. Quarto rendering, PDF generation, and the
plotting helpers sit at different external boundaries, so they are better
understood by reading their small wrappers and trying them in the local
environment.

For the normal development loop, the repository's own tools make the source
easy to inspect:

```console
.venv/bin/wt cat nb/notes/005-wt-src-walkthrough --index 3 --context 1
.venv/bin/wt diff nb/notes/005-wt-src-walkthrough
.venv/bin/pytest -q
```


## 9. The compact mental model

The package becomes much easier to hold in your head if you group commands by
the kind of state they move through:

```{mermaid}
flowchart TB
    command["wt command"] --> notebook_flow["resolve -> read -> render or mutate -> write"]
    command --> course_flow["resolve course -> find tagged cells -> format or encode"]
    command --> kernel_flow["start kernel -> execute -> persist outputs"]
    command --> external_flow["call Quarto, LaTeX, uv, or keyring"]
```

From that model, the source-reading order is natural:

1. `cli.py` tells you what the user can ask for.
2. `inspect.py` tells you how a name becomes a path.
3. `notebook.py` tells you how cells become readable text and how edits are
   written.
4. `execute.py` and `outputs.py` explain the lifecycle of stored results.
5. `problems.py` shows how tags turn ordinary notebooks into course content.
6. `scaffold.py`, `convert.py`, and the external wrappers fill in the rest of
   the repository workflow.

The recurring design choice is simple: keep the CLI close to the shell, keep
the notebook as the canonical artifact, and put conventions in small modules
that can be tested with temporary files. Once that is clear, the source is no
longer a list of unrelated commands. It is a set of short, inspectable paths
through the same filesystem-backed knowledge base.
